# 02. Preprocesamiento sin fuga de información

Preparamos las reservas de los dos hoteles para estimar el riesgo de cancelación **al registrar una reserva**. Partimos del diagnóstico del notebook 01. Aplicamos reglas fijas de calidad, seleccionamos las variables disponibles en ese momento y separamos los datos por perfil de reserva. Dejamos las transformaciones que aprenden de los datos dentro del pipeline de cada modelo.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from sklearn.pipeline import Pipeline

sys.path.insert(0, str(Path("../src").resolve()))

from preprocesamiento import (
    COLUMNAS_CATEGORICAS,
    COLUMNAS_NUMERICAS,
    COLUMNAS_PREDICTORAS,
    FECHAS,
    FRECUENCIA_MINIMA,
    GRUPO,
    N_PARTICIONES,
    OBJETIVO,
    crear_particionador,
    crear_preprocesador,
)

RUTA_CRUDA = Path("../data/raw/hotel_bookings.csv.zip")
RUTA_PROCESADA = Path("../data/processed")
ADR_MAXIMO = 1_000
LIMITE_MENORES = 9
MESES = {
    mes: numero
    for numero, mes in enumerate(
        ("January", "February", "March", "April", "May", "June",
         "July", "August", "September", "October", "November", "December"),
        start=1,
    )
}
ENTRADAS = list(COLUMNAS_PREDICTORAS)
COLUMNAS_EXPORTADAS = [*ENTRADAS, OBJETIVO, GRUPO, *FECHAS]

reservas = pd.read_csv(RUTA_CRUDA)
assert set(reservas[OBJETIVO].unique()) == {0, 1}

## 1. Referencia antes de limpiar

Recalculamos la comparación de Julián con todas las columnas originales. Una fila idéntica puede ser una copia o representar otra habitación del mismo grupo, porque los datos carecen de un identificador de reserva. La comparación muestra cuánto pesa esa decisión en la población que termina representando el modelo.

In [2]:
def resumir_base(etapa, indices):
    """Resume la cancelación general y el peso de Non Refund en las filas conservadas."""
    datos = reservas.loc[indices]
    sin_reembolso = datos["deposit_type"].eq("Non Refund")
    return {
        "etapa": etapa,
        "filas": len(datos),
        "canceladas": int(datos[OBJETIVO].sum()),
        "cancelación (%)": datos[OBJETIVO].mean() * 100,
        "filas Non Refund": int(sin_reembolso.sum()),
        "peso Non Refund (%)": sin_reembolso.mean() * 100,
        "cancelación Non Refund (%)": datos.loc[sin_reembolso, OBJETIVO].mean() * 100,
    }


referencias = [
    resumir_base("Datos crudos", reservas.index),
    resumir_base("Sin duplicados en las 32 columnas", reservas.index[~reservas.duplicated()]),
]
display(pd.DataFrame(referencias).set_index("etapa").round(2))

cruda, unica = referencias
retiradas_non_refund = cruda["filas Non Refund"] - unica["filas Non Refund"]
print(f"Non Refund: se retiran {retiradas_non_refund:,} de {cruda['filas Non Refund']:,} filas "
      f"({retiradas_non_refund / cruda['filas Non Refund']:.1%}) con la deduplicación cruda.")

,filas,canceladas,cancelación (%),filas Non Refund,peso Non Refund (%),cancelación Non Refund (%)
etapa,,,,,,
Datos crudos,119390,44224,37.04,14587,12.22,99.36
Sin duplicados en las 32 columnas,87396,24025,27.49,1038,1.19,94.70


Non Refund: se retiran 13,549 de 14,587 filas (92.9%) con la deduplicación cruda.


Al quitar duplicados exactos, la cancelación baja de **37,0 % a 27,5 %**. Se retira el **92,9 %** de las filas `Non Refund` y esta categoría pasa de representar el **12,2 % al 1,2 %** de las reservas. Su tasa de cancelación sigue alta. Para interpretar el riesgo hay que distinguir la tasa dentro del grupo de su peso en toda la base.

## 2. Calidad de los registros y variables derivadas

Aplicamos las reglas propuestas en el diagnóstico. Tratamos como inválidas las tarifas negativas o mayores a 1.000, las reservas sin huéspedes, las de cero noches y los registros con nueve o más niños o bebés. El límite de tarifa separa los errores revisados en el EDA. Lo usamos para limpiar el histórico. La tarifa calculada en ese histórico queda fuera del modelo.

Conservamos los atípicos plausibles, como estancias largas y reservas sin adultos pero con menores. Si una fila incumple varias reglas, la contamos una sola vez, en la primera que se aplica.

In [3]:
base = reservas.copy()
base["children"] = base["children"].fillna(0).astype(int)
base["total_noches"] = base["stays_in_week_nights"] + base["stays_in_weekend_nights"]
base["total_huespedes"] = base["adults"] + base["children"] + base["babies"]

reglas = {
    "Tarifa fuera de rango": ~base["adr"].between(0, ADR_MAXIMO),
    "Sin huéspedes": base["total_huespedes"].eq(0),
    "Sin noches": base["total_noches"].eq(0),
    "Conteo extremo de niños o bebés": (
        base["children"].ge(LIMITE_MENORES) | base["babies"].ge(LIMITE_MENORES)
    ),
}
auditoria = []
for motivo, invalida in reglas.items():
    indices_excluidos = base.index[invalida.loc[base.index]]
    base = base.drop(index=indices_excluidos)
    auditoria.append({"regla": motivo, "filas retiradas": len(indices_excluidos), "filas restantes": len(base)})

display(pd.DataFrame(auditoria).set_index("regla"))
referencia_valida = resumir_base("Después de filtros de calidad", base.index)

,filas retiradas,filas restantes
regla,,
Tarifa fuera de rango,2,119388
Sin huéspedes,180,119208
Sin noches,645,118563
Conteo extremo de niños o bebés,3,118560


Los conteos son secuenciales. Una reserva sin huéspedes y sin noches aparece únicamente en la primera regla que la excluye.

Para los faltantes usamos el significado de cada campo. Un `children` vacío se toma como cero según el tratamiento acordado. En `agent` usamos la categoría `Sin agente`, y en lugar de codificar cientos de empresas guardamos `tiene_empresa`. Unimos `Undefined` con `SC` únicamente en `meal`, donde ambas categorías representan ausencia de plan de comidas.

In [4]:
base["agent"] = base["agent"].astype("Int64").astype("string").fillna("Sin agente")
base["tiene_empresa"] = base["company"].notna().astype("int8")
base["meal"] = base["meal"].replace({"Undefined": "SC"})
base["mes_llegada"] = base["arrival_date_month"].map(MESES)

base["fecha_llegada"] = pd.to_datetime({
    "year": base["arrival_date_year"],
    "month": base["mes_llegada"],
    "day": base["arrival_date_day_of_month"],
})
base["fecha_registro_estimada"] = base["fecha_llegada"] - pd.to_timedelta(base["lead_time"], unit="D")

assert not base[[*ENTRADAS, OBJETIVO, *FECHAS]].isna().any().any()

## 3. Qué sabemos al registrar la reserva

Tomamos las recomendaciones del EDA y contrastamos las definiciones con la [tabla 1 del artículo de Antonio, de Almeida y Nunes](https://pmc.ncbi.nlm.nih.gov/articles/PMC6297060/#tbl1). Los autores construyeron el histórico con un punto de observación previo a la llegada. Su diccionario permite distinguir el dato de la reserva de campos calculados más tarde.

| Variables | Decisión | Motivo |
|---|---|---|
| `reservation_status`, `reservation_status_date` | Fuera | El estado y su fecha revelan el desenlace. |
| `assigned_room_type`, `booking_changes`, `days_in_waiting_list` | Fuera | La asignación, los cambios acumulados y la espera final se conocen después del registro. |
| `country`, `required_car_parking_spaces`, `total_of_special_requests` | Fuera | La captura inicial es incierta. El país puede corregirse al llegar. El EDA encontró cero cancelaciones con parqueadero, y los pedidos se pueden añadir más tarde. |
| `deposit_type`, `adr` | Fuera | El depósito de este CSV se clasifica usando pagos registrados hasta la llegada o cancelación. La tarifa diaria se calcula a partir de transacciones de alojamiento. Usamos el depósito solo para el diagnóstico de duplicados. |
| `hotel`, `lead_time`, `mes_llegada`, `reserved_room_type`, `meal` | Entran | Describen hotel, anticipación, temporada, habitación solicitada y plan de comidas. El mes se trata como categoría. |
| `market_segment`, `distribution_channel`, `customer_type`, `agent`, `tiene_empresa` | Entran | Describen cómo se registra y quién gestiona la reserva. Los códigos de agencia son categorías. |
| `total_noches`, `stays_in_weekend_nights`, `total_huespedes`, `children`, `babies` | Entran | Describen duración y composición. El total y una parte de las noches conservan la distinción entre semana y fin de semana. El total, niños y bebés conservan la composición de huéspedes. |
| `is_repeated_guest`, `previous_cancellations`, `previous_bookings_not_canceled` | Entran | Describen el historial previo a la reserva según el diccionario. |
| `fecha_llegada`, `fecha_registro_estimada`, `grupo_perfil` | Solo control | Permiten auditar fechas y mantener juntos los perfiles durante la partición. Del calendario crudo dejamos solo el mes como predictor. |
| `is_canceled` | Objetivo | Se usa para estratificar y como etiqueta en el componente supervisado. |

En una aplicación real necesitaríamos la versión inicial de los campos que admiten cambios, como el número de huéspedes, las noches o la habitación solicitada. El histórico carece de esas versiones. Usamos estas características como aproximación a los datos iniciales y mantenemos esa limitación al interpretar la evaluación.

## 4. Duplicados sobre lo que verá el modelo

Quitamos duplicados **después** de aplicar las transformaciones fijas y elegir los predictores. Al retirar campos o reemplazar los códigos de empresa por un indicador, algunas filas antes distintas pasan a verse iguales.

Conservamos la primera fila del CSV cuando coinciden el perfil, la fecha de llegada y la etiqueta. Si cambia la fecha, preservamos la observación para el análisis temporal. Si cambia la etiqueta, conservamos los dos resultados. En ambos casos las reservas con los mismos predictores quedan en el mismo grupo, aunque difieran en su fecha o resultado.

La deduplicación reduce el peso de perfiles frecuentes. Sin un identificador de reserva no podemos saber cuáles coincidencias son copias y cuáles representan habitaciones distintas.

In [5]:
claves = [*ENTRADAS, "fecha_llegada", OBJETIVO]
columnas_limpias = [*ENTRADAS, OBJETIVO, *FECHAS]
filas_sin_fecha = len(base.drop_duplicates(subset=[*ENTRADAS, OBJETIVO]))
base = base[columnas_limpias].drop_duplicates(subset=claves).copy()
base[GRUPO] = base.groupby(ENTRADAS, sort=False, dropna=False).ngroup()

resumen = pd.DataFrame([
    *referencias,
    referencia_valida,
    resumir_base("Base final por perfil, fecha y etiqueta", base.index),
]).set_index("etapa")
display(resumen.round(2))

por_perfil = base.groupby(GRUPO)[OBJETIVO].nunique()
perfiles_mixtos = por_perfil.gt(1).sum()
print(f"Al distinguir las fechas conservamos {len(base) - filas_sin_fecha:,} filas adicionales.")
print(f"Quedan {len(por_perfil):,} perfiles distintos, de los cuales {perfiles_mixtos:,} tienen ambas etiquetas.")
assert not base.duplicated(subset=claves).any()

,filas,canceladas,cancelación (%),filas Non Refund,peso Non Refund (%),cancelación Non Refund (%)
etapa,,,,,,
Datos crudos,119390,44224,37.04,14587,12.22,99.36
Sin duplicados en las 32 columnas,87396,24025,27.49,1038,1.19,94.70
Después de filtros de calidad,118560,44174,37.26,14586,12.30,99.36
"Base final por perfil, fecha y etiqueta",79190,23139,29.22,966,1.22,95.45


Al distinguir las fechas conservamos 3,857 filas adicionales.
Quedan 73,189 perfiles distintos, de los cuales 2,144 tienen ambas etiquetas.


La base final conserva 79.190 filas y tiene un 29,22 % de cancelación. La referencia del 27,5 % solo quita duplicados exactos del CSV. Aquí aplicamos filtros de calidad, simplificamos variables y comparamos perfiles con la misma fecha. Entre las filas conservadas, 966 estaban registradas como `Non Refund` en el crudo, el 1,22 % de la base. Ese conteo depende de cuál fila se conserva cuando coincide la clave.

## 5. Partición por perfil con estratificación

Usamos `StratifiedGroupKFold` con semilla 42 y cinco particiones. Reservamos la primera quinta parte para prueba y el resto para entrenamiento. El método intenta mantener la proporción de cancelaciones con grupos indivisibles, por lo que la relación 80/20 y el balance de clases son aproximados. Fijamos el primer bloque de antemano, sin evaluar modelos para escogerlo.

La clave de grupo también queda disponible para que la validación cruzada posterior mantenga los perfiles juntos dentro de entrenamiento. Así evitamos que dos filas con los mismos predictores se repartan entre ajuste y evaluación, incluso cuando sus etiquetas difieren.

In [6]:
indices_entrenamiento, indices_prueba = next(crear_particionador().split(
    base[ENTRADAS], base[OBJETIVO], groups=base[GRUPO]
))
entrenamiento = base.iloc[indices_entrenamiento][COLUMNAS_EXPORTADAS].copy()
prueba = base.iloc[indices_prueba][COLUMNAS_EXPORTADAS].copy()

assert entrenamiento.index.intersection(prueba.index).empty
assert len(entrenamiento) + len(prueba) == len(base)
assert set(entrenamiento[GRUPO]).isdisjoint(prueba[GRUPO])
assert len(entrenamiento[ENTRADAS].merge(prueba[ENTRADAS].drop_duplicates(), on=ENTRADAS)) == 0

particiones = pd.DataFrame([
    {
        "partición": nombre,
        "filas": len(datos),
        "porción de la base (%)": len(datos) / len(base) * 100,
        "canceladas": int(datos[OBJETIVO].sum()),
        "cancelación (%)": datos[OBJETIVO].mean() * 100,
        "perfiles": datos[GRUPO].nunique(),
    }
    for nombre, datos in (("Entrenamiento", entrenamiento), ("Prueba", prueba))
]).set_index("partición")
display(particiones.round(2))
print("Perfiles compartidos entre entrenamiento y prueba: 0")

,filas,porción de la base (%),canceladas,cancelación (%),perfiles
partición,,,,,
Entrenamiento,63351,80.0,18511,29.22,58552
Prueba,15839,20.0,4628,29.22,14637


Perfiles compartidos entre entrenamiento y prueba: 0


## 6. Transformaciones dentro del pipeline

Estandarizamos las numéricas para que sus escalas sean comparables. Codificamos las categóricas con one-hot para evitar un orden artificial. El codificador agrupa categorías con menos de 20 observaciones dentro del conjunto con el que se ajusta. Es un umbral fijo para controlar la cardinalidad, no un valor elegido con métricas de prueba. Las categorías nuevas se envían al grupo infrecuente cuando existe. En caso contrario se representan con ceros en las columnas de esa variable.

Comprobamos el preprocesador con un fold interno de entrenamiento. Ajustamos sobre las filas de ajuste y aplicamos el mismo transformador a las filas de validación. En los notebooks siguientes se crea un pipeline nuevo con `crear_preprocesador()` y el estimador. Se entregan al pipeline las columnas sin escalar ni codificar, y se usa `crear_particionador()` con `grupo_perfil` para obtener los folds.

In [7]:
X_entrenamiento = entrenamiento[ENTRADAS]
y_entrenamiento = entrenamiento[OBJETIVO]
grupos_entrenamiento = entrenamiento[GRUPO]
folds = list(crear_particionador().split(X_entrenamiento, y_entrenamiento, grupos_entrenamiento))
assert len(folds) == N_PARTICIONES
for ajuste, validacion in folds:
    assert set(grupos_entrenamiento.iloc[ajuste]).isdisjoint(grupos_entrenamiento.iloc[validacion])

ajuste, validacion = folds[0]
X_ajuste = X_entrenamiento.iloc[ajuste]
X_validacion = X_entrenamiento.iloc[validacion]
pipeline = Pipeline([("preprocesamiento", crear_preprocesador())])
Z_ajuste = pipeline.fit_transform(X_ajuste)
Z_validacion = pipeline.transform(X_validacion)

transformador = pipeline.named_steps["preprocesamiento"]
escalador = transformador.named_transformers_["numericas"]
codificador = transformador.named_transformers_["categoricas"]
np.testing.assert_allclose(escalador.mean_, X_ajuste[list(COLUMNAS_NUMERICAS)].mean())
assert Z_ajuste.shape[1] == Z_validacion.shape[1]
posicion_agente = COLUMNAS_CATEGORICAS.index("agent")
agentes_raros = codificador.infrequent_categories_[posicion_agente]
print(f"Ajuste: {Z_ajuste.shape[0]:,} filas y {Z_ajuste.shape[1]} columnas transformadas.")
print(f"Validación: {Z_validacion.shape[0]:,} filas con las mismas {Z_validacion.shape[1]} columnas.")
print(f"Agencias por debajo del umbral de {FRECUENCIA_MINIMA}: {len(agentes_raros)} en el fold de ajuste.")

Ajuste: 50,680 filas y 153 columnas transformadas.
Validación: 12,671 filas con las mismas 153 columnas.
Agencias por debajo del umbral de 20: 212 en el fold de ajuste.


El escalador usa las medias del fold de ajuste y validación recibe las mismas columnas codificadas. El número de agencias que se agrupan depende de qué filas estén disponibles en el ajuste. Las comprobaciones anteriores revisan la preparación. La evaluación del clasificador corresponde al notebook 03.

## 7. Exportación y uso

Guardamos dos CSV comprimidos con las columnas limpias, la etiqueta, el grupo y las fechas de control. Se pueden leer directamente con `pd.read_csv`. Las columnas de `COLUMNAS_PREDICTORAS` forman `X`, e `is_canceled` forma `y`. Para validación cruzada se pasa `grupo_perfil` al particionador.

La fecha de registro se estima restando la anticipación a la fecha de llegada. Conservamos ambas fechas para la validación temporal posterior y mantenemos separadas las fechas y el grupo de las entradas del modelo. Para segmentación, se seleccionan variables de perfil sin incorporar `is_canceled` al ajuste. Se puede recuperar la población limpia concatenando las particiones.

In [8]:
RUTA_PROCESADA.mkdir(parents=True, exist_ok=True)
for nombre, datos in (("entrenamiento", entrenamiento), ("prueba", prueba)):
    ruta = RUTA_PROCESADA / f"{nombre}.csv.gz"
    datos.to_csv(ruta, index=False, compression={"method": "gzip", "mtime": 0})
    print(f"{ruta.name}: {len(datos):,} filas")

tasa_base = base[OBJETIVO].mean() * 100
tasa_ajuste = entrenamiento[OBJETIVO].mean() * 100
tasa_prueba = prueba[OBJETIVO].mean() * 100
display(Markdown(
    f"La base final conserva **{len(base):,} filas** con **{tasa_base:.2f} %** de cancelación. "
    f"Entrenamiento queda en **{tasa_ajuste:.2f} %** y prueba en **{tasa_prueba:.2f} %**. "
    "Las dos particiones conservan una prevalencia muy cercana y tienen perfiles disjuntos. "
    "Estas tasas describen las reservas conservadas tras la limpieza y la deduplicación."
))

La base final conserva **79,190 filas** con **29.22 %** de cancelación. Entrenamiento queda en **29.22 %** y prueba en **29.22 %**. Las dos particiones conservan una prevalencia muy cercana y tienen perfiles disjuntos. Estas tasas describen las reservas conservadas tras la limpieza y la deduplicación.

entrenamiento.csv.gz: 63,351 filas
prueba.csv.gz: 15,839 filas
